# Spark Lakehouse — Retail Data Unification
## Bronze Layer: Raw Data Ingestion

### 1. Business Context

The retail business operates two independent systems: an online commerce platform and an offline Point-of-Sale (POS) system. Each system maintains separate customer, product, and order datasets, resulting in inconsistent schemas and fragmented data.

This project implements a Spark-based Lakehouse pipeline on Databricks, using Amazon S3 for source storage and Delta Lake for managed analytical tables.

### 2. Objective

The Bronze layer establishes the raw ingestion foundation by loading data from both source systems into Delta tables while retaining the source attributes and adding ingestion metadata.

### 3. Scope

- Ingest online and offline customer, product, and order datasets from Amazon S3.
- Process date-organized source files using a configurable run-date widget.
- Add metadata to support batch identification and data lineage.
- Persist six source-specific Delta tables in the Bronze schema.
- Support reruns for a selected file date.

### 4. Medallion Architecture

| Layer | Responsibility |
|---|---|
| **Bronze** — Current notebook | Raw ingestion and metadata capture |
| **Silver** — Next stage | Data cleansing, standardization, deduplication, and entity unification |
| **Gold** — Final stage | Dimensional modelling and analytics-ready data |

### 5. Technology Stack

**Apache Spark · PySpark · Azure Databricks · Amazon S3 · Delta Lake · SQL**

### 6. Expected Outcome

Six Bronze Delta tables containing online and offline customer, product, and order data, with ingestion metadata available for downstream processing and traceability.

**Design principle:** The Bronze layer focuses on ingestion and traceability. Business-level cleansing and unification are handled in subsequent layers.

## 1. Configuration and Batch Selection

The notebook uses a Databricks widget named `file_run_date` to select the date folder containing the source files.

- **Default value:** Yesterday's date, formatted as `YYYYMMDD`.
- **Purpose:** Allows the same notebook to process different daily batches without hardcoding the date in each source path.
- **Operational consideration:** Confirm the selected date before execution to ensure that the intended source batch is processed.

This configuration makes the ingestion process reusable across daily file arrivals.

In [0]:
from datetime import date, timedelta
from pyspark.sql.functions import *

yesterday = date.today() - timedelta(days=1)
yesterday_str = yesterday.strftime('%Y%m%d')
#print(yesterday_str)

dbutils.widgets.text("file_run_date", yesterday_str)
#dbutils.widgets.text("file_run_date", "20260207") #as running with data as of 20260207
file_run_date = dbutils.widgets.get("file_run_date")

#dbutils.widgets.remove("file_run_date")
print(file_run_date)

20260207


## 2. Rerun Handling and Batch Replacement

### Why is this required?

A pipeline may need to be rerun because of a failed execution, a corrected source file, or a request to reload a specific daily batch. Without appropriate handling, rerunning the same batch could create duplicate records in the Bronze tables.

### Approach

The notebook deletes existing records for the selected `file_run_date` before reloading that batch. Data from other file dates remains untouched.

The notebook also includes a Python-based alternative to the SQL deletion logic for demonstration purposes. Only the intended deletion approach should execute during a run.

**Important:** This is a date-specific replacement strategy. It supports rerunning the same date, but it does not by itself provide an atomic transaction across all six tables.

In [0]:
%sql
delete from `spark-databricks-project-catalog`.bronze.online_orders where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.online_customers where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.online_products where file_date = :file_run_date;

delete from `spark-databricks-project-catalog`.bronze.offline_orders where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.offline_customers where file_date = :file_run_date;
delete from `spark-databricks-project-catalog`.bronze.offline_products where file_date = :file_run_date;

num_affected_rows
15000


## The above delete can be done in Python too

In [0]:
%skip
tables = ['online_orders', 'online_customers', 'online_products', 'offline_orders', 'offline_customers', 'offline_products']

for table in tables:
    spark.sql(f"delete from `spark-databricks-project-catalog`.bronze.{table} where file_date = '{file_run_date}'")

## 3. Online System — Source Ingestion

The online commerce platform provides three datasets: customers, products, and orders. These datasets are stored in date-organized folders in Amazon S3.

### Processing steps

1. Read the source CSV files from the selected date folder.
2. Add the file-date metadata to identify the ingestion batch.
3. Capture ingestion timestamp and source-system information.
4. Write the resulting DataFrames to their corresponding Bronze Delta tables.

### Target tables

| Dataset | Bronze table |
|---|---|
| Customer data | `bronze.online_customers` |
| Product data | `bronze.online_products` |
| Order data | `bronze.online_orders` |

These tables retain source-specific data for subsequent cleansing, customer matching, product unification, and order enrichment in the Silver layer.

In [0]:
#orders table
df = spark.read.format("csv") \
     .option("header", True) \
     .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_orders.csv")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())
#df_audit.printSchema()

df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.online_orders")

#df = spark.table("`spark-databricks-project-catalog`.bronze.online_orders") 
#df.show()       

In [0]:
#customers table
df = spark.read.format("csv") \
               .option("header", True) \
               .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_customers*")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.online_customers") 

#df = spark.table("`spark-databricks-project-catalog`.bronze.online_customers") 
#df.show(10)                              

In [0]:
#products table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/online data/{file_run_date}/online_products*")

df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("online")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.online_products")    
                   
#df = spark.table("`spark-databricks-project-catalog`.bronze.online_products")
#df.show(10)


## 4. Offline System — Source Ingestion

The offline retail system provides customer, product, and order datasets generated by the store Point-of-Sale (POS) system.

The ingestion process follows the same Bronze-layer principles as the online source while using the offline system's own file paths and schemas.

### Target tables

| Dataset | Bronze table |
|---|---|
| Customer data | `bronze.offline_customers` |
| Product data | `bronze.offline_products` |
| Order data | `bronze.offline_orders` |

Keeping online and offline data in separate Bronze tables preserves source lineage and allows differences between the two systems to be resolved systematically in the Silver layer.

In [0]:
#offline orders table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_orders.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("offline")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_orders")

# df = spark.table("`spark-databricks-project-catalog`.bronze.offline_orders")
# df.show(10)
                       

In [0]:
#customer table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_customers.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source",lit("offline")) \
             .withColumn("ingest_ts",current_timestamp())

df_audit.write.mode("append") \
              .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_customers")

# df = spark.table("`spark-databricks-project-catalog`.bronze.offline_customers")  
# df.show(10)           

In [0]:
#products table
df = spark.read.format("csv") \
               .option("header",True) \
               .load(f"s3://spark-databricks-project/orders/offline data/{file_run_date}/offline_products.csv")
              
df_audit = df.withColumn("file_date",lit(file_run_date)) \
             .withColumn("source", lit("offline")) \
             .withColumn("ingest_ts", current_timestamp())
             
df_audit.write.mode("append") \
        .saveAsTable("`spark-databricks-project-catalog`.bronze.offline_products")               

## 5. Bronze Layer — Output Summary

Upon successful execution, the notebook populates six source-specific Delta tables:

- `bronze.online_customers`
- `bronze.online_products`
- `bronze.online_orders`
- `bronze.offline_customers`
- `bronze.offline_products`
- `bronze.offline_orders`

### Metadata captured

| Column | Purpose |
|---|---|
| `file_date` | Identifies the source batch date |
| `source_system` | Identifies whether the data originated from the online or offline system |
| `ingestion_timestamp` | Records when the data was ingested |

### Downstream usage

The Silver layer will standardize schemas, clean customer and product attributes, unify entities across systems, and enrich orders. The Gold layer will use the processed data to build dimensions and a fact table for analytics.

**Outcome:** A source-preserving Bronze foundation for the Retail Data Unification Lakehouse pipeline.